# 02 — Fixed foveation (HOOK A)

Foveation imitates the human eye: sharp at the centre of gaze, degraded in
the periphery. Applied to a robot policy's input it keeps the centre and
throws away peripheral information.

**"Fixed"** means the fovea sits at the image centre and stays there. There
is no tracking, no gaze prediction, no privileged state. That is what makes
it comparable across backbones and benchmarks — the transform is a pure
function of the frame.

The code below is **copied verbatim** from
`adaptive_sparse_vla/foveation.py`, which is itself a bit-identical port of
the original RetinaBased OpenVLA implementation (verified by 29 checks
across 3 image sizes × 4 keep ratios). Copying rather than importing keeps
this notebook self-contained.

## Two variants, and why the difference matters

| | `blur` | `logpolar` |
|---|---|---|
| what it does | progressively **blurs** with distance from the centre | **resamples**: dense at the centre, sparse outward |
| pixel positions | **unchanged** | **moved** (periphery pulled inward) |
| what is lost | peripheral sharpness | peripheral spatial resolution |

`keep_ratio` has the same meaning in both: the effective sampling density
(log-polar) or the fully-sharp area (blur) is that fraction of the frame.
`keep_ratio=0.20` is the setting used throughout.

**This distinction is not cosmetic, and which variant is safe depends on the
backbone.** Any policy whose visual tokens carry meaning tied to *where* a
pixel was — a position embedding indexed by patch coordinate, depth
back-projected through camera intrinsics, an explicit 3D position per token
— is damaged by a transform that displaces pixels, because after warping
every token is stamped with the wrong location. A policy that treats the
image as appearance features without positional grounding is not.

SpatialVLA is the instance we measured: it back-projects each patch's *grid
coordinate* through the intrinsics, and log-polar cost it −7.3 points on
SimplerEnv Bridge while blur, which displaces nothing, recovered most of it.

**Check this on any new backbone before choosing a variant.** If the model
computes anything from patch coordinates, `blur` is the honest choice;
running only log-polar on such a model measures the warp, not foveation.

In [ ]:
import math
from typing import Optional, Tuple

import cv2
import numpy as np


def _uniform_sample_grid(height, width, keep_ratio):
    keep_ratio = float(np.clip(keep_ratio, 0.0, 1.0))
    if keep_ratio <= 0.0:
        return np.array([0], dtype=np.int32), np.array([0], dtype=np.int32)
    sample_scale = math.sqrt(keep_ratio)
    sample_rows = max(1, int(round(height * sample_scale)))
    sample_cols = max(1, int(round(width * sample_scale)))
    ys = np.linspace(0, height - 1, num=sample_rows, dtype=np.int32)
    xs = np.linspace(0, width - 1, num=sample_cols, dtype=np.int32)
    return ys, xs


def foveate_image_logpolar(image, keep_ratio, center=None):
    """Warp to log-polar, subsample uniformly there, warp back.

    Sampling uniformly in log-polar space is what produces the radial
    density falloff: equal steps in log-radius are small steps near the
    pole and large ones far from it.
    """
    frame = np.asarray(image, dtype=np.uint8)
    if frame.ndim != 3 or frame.shape[2] != 3:
        raise ValueError(f"Expected HxWx3 image, got {frame.shape}")
    if keep_ratio <= 0.0:
        return np.zeros_like(frame)

    height, width = frame.shape[:2]
    if center is None:
        center = (width / 2.0, height / 2.0)
    else:
        center = (float(np.clip(center[0], 0, width - 1)),
                  float(np.clip(center[1], 0, height - 1)))
    max_radius = float(np.hypot(max(center[0], width - center[0]),
                                max(center[1], height - center[1])))
    fwd = cv2.INTER_LINEAR + cv2.WARP_FILL_OUTLIERS + cv2.WARP_POLAR_LOG
    inv = fwd + cv2.WARP_INVERSE_MAP

    logpolar = cv2.warpPolar(frame, (width, height), center, max_radius, fwd)
    ys, xs = _uniform_sample_grid(height, width, keep_ratio)
    sampled = logpolar[np.ix_(ys, xs)]
    interpolated = cv2.resize(sampled, (width, height), interpolation=cv2.INTER_LINEAR)
    restored = cv2.warpPolar(interpolated, (width, height), center, max_radius, inv)
    return np.asarray(np.clip(restored, 0, 255), dtype=np.uint8)


def foveate_image_blur(image, keep_ratio, center=None):
    """Geometry-preserving foveation: sharp disc, blurred surround, no warp.

    A disc whose area is ~keep_ratio of the frame stays bit-identical;
    outside it, three blur levels are blended with radial weights. Every
    output pixel keeps its input coordinates.
    """
    frame = np.asarray(image, dtype=np.uint8)
    if frame.ndim != 3 or frame.shape[2] != 3:
        raise ValueError(f"Expected HxWx3 image, got {frame.shape}")

    keep_ratio = float(keep_ratio)
    height, width = frame.shape[:2]
    if keep_ratio >= 1.0:
        return frame.copy()

    if center is None:
        center = (width / 2.0, height / 2.0)
    else:
        center = (float(np.clip(center[0], 0, width - 1)),
                  float(np.clip(center[1], 0, height - 1)))

    r0 = math.sqrt(max(keep_ratio, 0.0) * height * width / math.pi)
    max_radius = float(np.hypot(max(center[0], width - center[0]),
                                max(center[1], height - center[1])))
    ramp = max(max_radius - r0, 1e-6)

    ys, xs = np.mgrid[0:height, 0:width]
    dist = np.hypot(xs - center[0], ys - center[1])
    t = np.clip((dist - r0) / ramp, 0.0, 1.0).astype(np.float32)

    blur_mid = cv2.GaussianBlur(frame, (0, 0), sigmaX=3.0)
    blur_far = cv2.GaussianBlur(frame, (0, 0), sigmaX=9.0)

    w_far = np.clip(2.0 * t - 1.0, 0.0, 1.0)[..., None]
    w_mid = np.clip(2.0 * t, 0.0, 1.0)[..., None] - w_far
    w_sharp = 1.0 - w_mid - w_far

    out = (frame.astype(np.float32) * w_sharp
           + blur_mid.astype(np.float32) * w_mid
           + blur_far.astype(np.float32) * w_far)
    out = np.clip(np.rint(out), 0, 255).astype(np.uint8)
    out[dist <= r0] = frame[dist <= r0]   # fovea exactly identical
    return out

## HOOK A — where this goes

On the **raw environment frame, before the policy's own preprocessing.**

```
obs = env.step(...)
image = get_image(obs)          # (H, W, 3) uint8, e.g. 256×256
image = foveate_image_blur(image, 0.20)      ← HERE
actions = policy.step(image, instruction)    # policy resizes to 224 itself
```

### Getting this wrong is easy and silent

| placement | consequence |
|---|---|
| **before** the policy's resize ✅ | correct: the policy sees a foveated scene at its normal input size |
| after the resize / inside the model | the transform operates on already-downsampled pixels, so `keep_ratio` no longer means what it says, and results are not comparable to any other backbone |
| on the normalised float tensor | `cv2` operations on a normalised tensor produce a valid-looking image that is not the intended transform |

Neither mistake raises an error. Both just lower the success rate, which is
indistinguishable from "the method does not work".

In [ ]:
def make_foveation_hook(mode="blur", keep_percent=20.0, views=None):
    """Returns an image_fn for `run_episode` in notebook 01.

    Handles whatever shape the adapter's `get_image` returns:

      * a single (H, W, 3) array          -> foveated
      * a dict   {"agent": arr, ...}      -> every entry foveated
      * a list/tuple of arrays            -> every element foveated

    `views` restricts a dict/sequence to named entries. Leaving it None
    degrades EVERY view, which is the condition reported so far -- see the
    note below on why foveating only one view measures the wrong thing.
    """
    fn = foveate_image_blur if mode == "blur" else foveate_image_logpolar
    keep_ratio = keep_percent / 100.0

    def _one(arr):
        return fn(np.asarray(arr), keep_ratio=keep_ratio, center=None)

    def image_fn(image, state):
        if isinstance(image, dict):
            return {k: (_one(v) if (views is None or k in views) else v)
                    for k, v in image.items()}
        if isinstance(image, (list, tuple)):
            return type(image)(
                _one(v) if (views is None or i in views) else v
                for i, v in enumerate(image))
        return _one(image)

    return image_fn


# usage with the loop from 01:
#   run_episode(adapter, policy, instruction,
#               image_fn=make_foveation_hook("blur", 20.0))

## What this assumes about the policy — check before porting

The transform itself is model-agnostic, but *wiring it in* is not. Four
assumptions are baked into the hook above; each one is a real difference
between VLAs, and each fails quietly rather than loudly.

| assumption | fails when | what to do |
|---|---|---|
| **one frame per step** | the policy consumes a *history* of frames (a window of the last 8–16 observations) | foveate every frame entering the window, not just the newest. Foveating one frame in a window of 8 measures a mixed input, and the effective intervention strength is 1/8 of what it says |
| **uint8 RGB, `H×W×3`** | the env hands back BGR, float, or CHW | convert before, convert back after. `cv2` reads and writes BGR by default; a silent channel swap changes colours the policy was trained on and looks like a method failure |
| **`keep_ratio` is relative to the frame it is given** | the env renders larger than the policy's input (e.g. 640×480 → 224) | the sharp disc is a fraction of *this* frame, so applying it at 640 then downsampling to 224 is not the same intervention as applying it at 256. Fix the resolution at which foveation happens and record it |
| **one camera** | the policy reads several views | the hook above already handles a dict or list of views and degrades all of them by default -- but you must still decide and record the choice, see below |

### More than one camera

LIBERO gives an agent view and a wrist view; some setups add more. Decide
explicitly and record the decision, because "foveate the agent view only"
and "foveate every view" are different conditions, and the gap between them
can be as large as the effect being measured.

Degrading only one view leaves the other as an unfoveated backup, which can
make a backbone look robust when it is merely reading the other camera. The
runs reported so far degrade **every view the policy receives**.

If a gaze-driven centre is ever used, note that a centre computed on the
agent view is meaningless on a wrist view — they see different scenes. Give
each view its own centre or leave the secondary views centred.

## Visual check

Always look at the output once before trusting a run. A transform that is
subtly wrong — fovea in the wrong place, keep ratio misinterpreted,
channels swapped — produces a plausible image and a quietly lower score.

In [ ]:
def _demo_frame(size=256):
    """Synthetic scene: a few coloured discs on a gradient background."""
    ys, xs = np.mgrid[0:size, 0:size]
    img = np.zeros((size, size, 3), dtype=np.uint8)
    img[..., 0] = (xs * 255 // size).astype(np.uint8)
    img[..., 1] = (ys * 255 // size).astype(np.uint8)
    img[..., 2] = 90
    for (cx, cy, r, col) in [(60, 70, 22, (255, 40, 40)),
                             (128, 128, 26, (40, 255, 90)),
                             (200, 190, 20, (60, 120, 255))]:
        m = (xs - cx) ** 2 + (ys - cy) ** 2 <= r * r
        img[m] = col
    # fine texture, so the loss of high-frequency detail is visible
    img[::4, :, :] = np.clip(img[::4, :, :].astype(int) + 45, 0, 255).astype(np.uint8)
    return img


frame = _demo_frame()
out_blur = foveate_image_blur(frame, 0.20)
out_lp = foveate_image_logpolar(frame, 0.20)

strip = np.concatenate([frame, out_blur, out_lp], axis=1)
cv2.imwrite("foveation_demo.png", cv2.cvtColor(strip, cv2.COLOR_RGB2BGR))
print("raw | blur 20% | log-polar 20%  ->  foveation_demo.png")

try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(13, 4.6))
    for i, (title, arr) in enumerate(
            [("raw", frame), ("blur 20%", out_blur), ("log-polar 20%", out_lp)]):
        plt.subplot(1, 3, i + 1); plt.imshow(arr); plt.title(title); plt.axis("off")
    plt.tight_layout(); plt.show()
except ImportError:
    pass

In [ ]:
# Properties worth asserting, not eyeballing.
centre_untouched = np.array_equal(
    foveate_image_blur(frame, 0.20)[120:136, 120:136],
    frame[120:136, 120:136])
print("blur leaves the fovea bit-identical:", centre_untouched)

print("blur moves no pixel (shape identical):",
      out_blur.shape == frame.shape)

def hf_energy(a):
    g = cv2.cvtColor(a, cv2.COLOR_RGB2GRAY).astype(np.float32)
    return float(np.abs(cv2.Laplacian(g, cv2.CV_32F)).mean())

print(f"high-frequency detail  raw {hf_energy(frame):6.2f}  "
      f"blur {hf_energy(out_blur):6.2f}  logpolar {hf_energy(out_lp):6.2f}")
print("\n-> both remove detail; only log-polar also displaces pixels.")

## The hook survives every observation shape

`get_image` returns whatever the benchmark's adapter gives it: one array on
a single-camera setup, a dict on LIBERO (agent + wrist), possibly a list.
The hook must not care -- otherwise it works on one benchmark and silently
skips views on another.

In [ ]:
single = _demo_frame(128)
multi_dict = {"agent": _demo_frame(128), "wrist": _demo_frame(96)}
multi_list = [_demo_frame(128), _demo_frame(96)]

hook_all = make_foveation_hook("blur", 20.0)

out = hook_all(single, {})
assert isinstance(out, np.ndarray) and out.shape == single.shape
print("single array  ->", out.shape)

out = hook_all(multi_dict, {})
assert set(out) == {"agent", "wrist"}
assert all(not np.array_equal(out[k], multi_dict[k]) for k in out)
print("dict of views ->", {k: v.shape for k, v in out.items()},
      "| both degraded")

out = hook_all(multi_list, {})
assert len(out) == 2 and all(not np.array_equal(a, b)
                             for a, b in zip(out, multi_list))
print("list of views ->", [v.shape for v in out], "| both degraded")

# Restricting to named views must leave the others bit-identical -- that is
# the "agent only" condition, and it is a DIFFERENT experiment.
hook_agent = make_foveation_hook("blur", 20.0, views={"agent"})
out = hook_agent(multi_dict, {})
assert not np.array_equal(out["agent"], multi_dict["agent"])
assert np.array_equal(out["wrist"], multi_dict["wrist"])
print("views={'agent'}  -> wrist left untouched (a different condition)")

print("\nhook is shape-agnostic")

## Caveats to carry into the results table

* **This does not reduce latency.** The image size is unchanged, so the
  policy processes the same number of visual tokens. Measured: 1882 → 1888
  ms on UniVLA, 524 → 518 ms on OpenVLA. Foveation is an accuracy
  intervention; if a speedup is wanted it has to come from somewhere else
  (see notebook 04).
* **Direction of effect is not universal.** On SimplerEnv Bridge, log-polar
  at 20% *helped* OpenVLA (+18.8) and UniVLA (+8.3) but *hurt* SpatialVLA
  (−7.3). On LIBERO the same code hurt both of the backbones it helped on
  Bridge. Expect the sign to depend on the scene, and do not assume a
  result transfers.
* **Fovea placement was tested and is probably not the explanation.**
  Placing the fovea exactly on the target using simulator ground truth
  (an upper bound no deployable gaze can beat) did not recover the loss on
  LIBERO — blur went 58% → 50%, i.e. no detectable improvement.